In [4]:
"""
Natural Gas Price Estimator
============================

Overview
--------
This module models month-end natural gas purchase prices (Oct 2020 -
Sep 2024) and exposes a single function, `estimate_price`, that returns
a price estimate for any date within that range or up to one year
beyond it.

Methodology
-----------
Exploratory analysis of the historical series reveals two dominant
components:

    1. Trend  - a gradual long-term increase in price over the
       four-year window.
    2. Seasonality - a recurring intra-year cycle, with prices rising
       toward winter (higher heating demand) and easing through the
       summer months.

To capture both, prices are modeled as a linear trend combined with a
Fourier seasonal component (two harmonics of an annual cycle), fit via
ordinary least squares:

    price(t) = a + b*t
             + c1*sin(2*pi*t/365.25) + c2*cos(2*pi*t/365.25)
             + c3*sin(4*pi*t/365.25) + c4*cos(4*pi*t/365.25)

where t is measured in days since the first observation. The first harmonic captures the dominant annual rise/fall; the second allows the
seasonal curve to deviate from a pure sine shape (e.g. a sharper winterpeak relative to the summer trough).

For dates that fall inside the historical window, the model defers todirect linear interpolation between the two nearest observed data
points, since real data is preferable to a smoothed approximation whenever it is available. The fitted trend + seasonal model is used
only to extrapolate beyond the last observed date, up to a one-year horizon.

Usage
-----
    from gas_price_model import estimate_price
    estimate_price("2024-12-15")
    estimate_price("2025-06-30")
"""

import numpy as np
import pandas as pd
from datetime import datetime

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
CSV_PATH = r"C:\Users\sherp\Downloads\Nat_Gas.csv"

# ---------------------------------------------------------------------
# Data ingestion and preprocessing
# ---------------------------------------------------------------------
_df = pd.read_csv(CSV_PATH)

# Parse date strings (e.g. "10/31/20") into proper datetime objects so
# that date arithmetic, comparisons, and ordering behave correctly.
_df["Dates"] = pd.to_datetime(_df["Dates"], format="%m/%d/%y")

# Ensure chronological order regardless of input file ordering.
_df = _df.sort_values("Dates").reset_index(drop=True)

# Boundaries of the observed dataset, used to classify any incoming
# date as in-sample (interpolation) or out-of-sample (extrapolation).
_FIRST_DATE = _df["Dates"].min()
_LAST_DATE = _df["Dates"].max()

# Per project scope, extrapolation is supported for up to one year
# beyond the most recent observation.
_MAX_EXTRAPOLATION_DATE = _LAST_DATE + pd.DateOffset(years=1)

# Represent dates as a continuous numeric axis (days since the first
# observation) to support regression and interpolation.
_t = (_df["Dates"] - _FIRST_DATE).dt.days.values.astype(float)
_y = _df["Prices"].values.astype(float)

# ---------------------------------------------------------------------
# Model definition: linear trend + annual seasonality (Fourier terms)
# ---------------------------------------------------------------------
def _design_matrix(t):
    """
    Construct the regression design matrix for the trend + seasonal
    model. Each column corresponds to one model term:

        [intercept, trend, sin(annual), cos(annual),
         sin(semi-annual), cos(semi-annual)]
    """
    omega = 2 * np.pi / 365.25
    return np.column_stack([
        np.ones_like(t),
        t,
        np.sin(omega * t),
        np.cos(omega * t),
        np.sin(2 * omega * t),
        np.cos(2 * omega * t),
    ])


# Fit model coefficients via ordinary least squares on the historical
# observations.
_X = _design_matrix(_t)
_coeffs, *_ = np.linalg.lstsq(_X, _y, rcond=None)


def _model_predict(t):
    """Evaluate the fitted trend + seasonal model at time offset(s) t."""
    X = _design_matrix(np.atleast_1d(t).astype(float))
    return X @ _coeffs


def _parse_date(date_input):
    """Normalize string or datetime input to a pandas Timestamp."""
    if isinstance(date_input, (datetime, pd.Timestamp)):
        return pd.Timestamp(date_input)
    return pd.to_datetime(date_input)


# ---------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------
def estimate_price(date_input):
    """
    Return an estimated natural gas purchase price for a given date.

    Parameters
    ----------
    date_input : str or datetime-like
        Target date (e.g. "2024-12-15", "12/15/24", or a
        datetime/Timestamp object).

    Returns
    -------
    float
        Estimated price, rounded to two decimal places.

    Notes
    -----
    - Dates within the observed range (31-Oct-2020 to 30-Sep-2024) are
      estimated via linear interpolation between the nearest two
      recorded prices.
    - Dates after the observed range, up to one year out, are
      estimated using the fitted trend + seasonal regression model.
    - Dates outside the supported range raise a ValueError, as
      reliable estimation cannot be guaranteed beyond it.
    """
    date = _parse_date(date_input)

    if date < _FIRST_DATE:
        raise ValueError(
            f"Date {date.date()} precedes the available data "
            f"({_FIRST_DATE.date()})."
        )
    if date > _MAX_EXTRAPOLATION_DATE:
        raise ValueError(
            f"Date {date.date()} exceeds the supported one-year "
            f"extrapolation horizon beyond the last observation "
            f"({_LAST_DATE.date()}). Maximum supported date: "
            f"{_MAX_EXTRAPOLATION_DATE.date()}."
        )

    t_days = (date - _FIRST_DATE).days

    if date <= _LAST_DATE:
        # In-sample: interpolate directly between observed data points.
        price = np.interp(t_days, _t, _y)
    else:
        # Out-of-sample: extrapolate using the fitted model.
        price = _model_predict(t_days)[0]

    return round(float(price), 2)


def estimate_price_range(start_date, end_date, freq="D"):
    """
    Generate price estimates across a date range, suitable for
    plotting or tabular review.

    Parameters
    ----------
    start_date, end_date : str or datetime-like
        Bounds of the desired range.
    freq : str
        Pandas frequency string (e.g. "D" for daily, "ME" for
        month-end).

    Returns
    -------
    pandas.DataFrame
        Columns: "Date", "EstimatedPrice".
    """
    dates = pd.date_range(_parse_date(start_date), _parse_date(end_date), freq=freq)
    prices = [estimate_price(d) for d in dates]
    return pd.DataFrame({"Date": dates, "EstimatedPrice": prices})


# ---------------------------------------------------------------------
# Validation / demonstration
# ---------------------------------------------------------------------
if __name__ == "__main__":
    test_dates = [
        "2020-10-31",   # earliest observation - exact match expected
        "2022-06-15",   # mid-range - interpolated estimate
        "2024-09-30",   # latest observation - exact match expected
        "2024-12-31",   # ~3 months extrapolated - seasonal winter rise expected
        "2025-06-30",   # ~9 months extrapolated - seasonal summer dip expected
        "2025-09-30",   # 1-year extrapolation boundary
    ]
    for d in test_dates:
        print(f"{d}: ${estimate_price(d)}")

2020-10-31: $10.1
2022-06-15: $10.55
2024-09-30: $11.8
2024-12-31: $12.99
2025-06-30: $12.1
2025-09-30: $12.44
